In [5]:
import boto3
import awswrangler as wr
import yaml
from pathlib import Path
import datetime
import os


In [15]:
# S3 data source Location
S3_BUCKET = 'cm-aws-s3-data-source'
S3_PATH = 'organization/sales'

#Credentials
#Get file location
parent_cwd = Path.cwd().parent
files = list(parent_cwd.glob(pattern="credentials.yml"))
#read yaml file
with open(str(files[0]), 'r') as f:
    credentials = yaml.safe_load(f)
    aws_access_key_id = credentials.get('aws').get('aws_access_key_id')
    aws_secret_access_key = credentials.get('aws').get('aws_secret_access_key')



In [11]:
#create an authorised session using boto3 + credentials
session = boto3.session.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)
# create s3 client to interact with AWS S3
s3 = session.client('s3')




Full Load - Snapshot

Full Load - Partition by Ingestion Date

In [ ]:
# HINTS
# 1. list_objects_v2 >> List all objects
# 2. download_file >> download file
# 3. Download all files to "destination/sales/<ingestion_date>"

In [17]:
#Get all s3 objects
s3_objects = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_PATH
)
s3_objects

{'ResponseMetadata': {'RequestId': 'DV5BAY6E0ZTPWHXB',
  'HostId': 'UBzm8vk0fggqyvZZ0U0obRBe1qPyzsEAK3NcF6ToHrtsxsuZyBvdCZBGwuX3N/m5Hxar3ndexffc+Bz6LYC2dYnQ0hvFpa/F',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'UBzm8vk0fggqyvZZ0U0obRBe1qPyzsEAK3NcF6ToHrtsxsuZyBvdCZBGwuX3N/m5Hxar3ndexffc+Bz6LYC2dYnQ0hvFpa/F',
   'x-amz-request-id': 'DV5BAY6E0ZTPWHXB',
   'date': 'Sat, 31 Jan 2026 13:03:46 GMT',
   'x-amz-bucket-region': 'ap-southeast-2',
   'content-type': 'application/xml',
   'transfer-encoding': 'chunked',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'IsTruncated': False,
 'Contents': [{'Key': 'organization/sales/',
   'LastModified': datetime.datetime(2025, 11, 29, 8, 29, 46, tzinfo=tzutc()),
   'ETag': '"d41d8cd98f00b204e9800998ecf8427e"',
   'ChecksumAlgorithm': ['CRC64NVME'],
   'ChecksumType': 'FULL_OBJECT',
   'Size': 0,
   'StorageClass': 'STANDARD'},
  {'Key': 'organization/sales/coffee_sales_202403.csv',
   'LastModified': datetime.datetime(2025, 11, 29, 8

In [18]:
#create destination folder
destination_folder = f"destination/sales/(datetime.datetime.now().strftime('%Y%m%d'))"
os.makedirs(destination_folder, exist_ok=True)

In [19]:
#read each object and it to destination folder
for obj in s3_objects['Contents']:
    #Check if it is a file
    if obj['Size'] > 0:
        # Get object data
        response = s3.get_object(Bucket=S3_BUCKET, Key=obj['Key'])
        #Extract file name
        filename = obj['Key'].split('/')[-1]
        # save file data to local
        with open(f"{destination_folder}/{filename}", "w") as f:
            f.write(response['Body'].read().decode('utf-8'))
